In [2]:
import os
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list

In [3]:
def normalize_dataframe(df, column_variants):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    mapped_cols = {}
    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[col] = std_col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    df_extra = df.select(schema_cols + extra_cols)

    return new_df, df_extra, extra_cols, missing_cols, mapped_cols

In [5]:
base_dir = os.getcwd()
file_path_excel = os.path.join(base_dir, "./../faker/messy_customer_data.xlsx")
file_path_csv = os.path.join(base_dir, "./../faker/messy_customer_data.csv")

In [6]:
excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
excel_df.to_csv(file_path_csv, index=False)

In [7]:
spark = SparkSession.builder.appName("NormalizeData").getOrCreate()
df = spark.read.csv(file_path_csv, header=True, inferSchema=True)

25/09/18 23:40:32 WARN Utils: Your hostname, KD4SH1-BFMG resolves to a loopback address: 127.0.1.1; using 192.168.100.4 instead (on interface wlan0)
25/09/18 23:40:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/18 23:40:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/18 23:40:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
new_df, extra_df, extra_cols, missing, mapped = normalize_dataframe(
    df, mapping_list.customer_mapping_dict
)

In [9]:
print("\nNormalized DataFrame:")
new_df.show(5)

print("\nDataFrame with Extra Columns:")
extra_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-------------------+----------------+--------------------+
|customer_id|     customer_name|customer_type|gender|      date_of_birth|   registration_date|customer_status|acquisition_channel|customer_segment|          created_at|
+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-------------------+----------------+--------------------+
|      10369|   Cynthia Gregory|          B2C|  Male|1971-10-04 00:00:00|2022-11-11 11:16:...|       Inactive|           LinkedIn|      High Value|2024-12-21 08:06:...|
|      10078|   Ms Karen Turner|       Guest*|Female|1994-04-30 00:00:00|2025-05-16 17:36:...|       Inactive|            Twitter|Occasional Buyer|2025-07-26 11:31:...|
|      10299|     Maureen Lewis|          B2C| Other|1996-07-11 00:00:00|2022-12-17 23:00:...|        Blocked|           Referral|  

25/09/18 23:40:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
